# 02 — Train LSTM

Run `00_setup_and_data.ipynb` first to set `ALL_NOTES`.


In [ ]:
import os
ALL_NOTES     = '/content/drive/MyDrive/deep-techno-data/all_notes.csv'
CHECKPOINT_DIR = '/content/drive/MyDrive/deep-techno-data/checkpoints/lstm'

## Config


In [ ]:
from deepTechno.training.train_lstm import LSTMConfig

config = LSTMConfig(
    all_notes_csv   = ALL_NOTES,
    checkpoint_dir  = CHECKPOINT_DIR,
    seq_length      = 25,
    vocab_size      = 128,
    batch_size      = 64,
    epochs          = 50,
    learning_rate   = 0.005,
    lstm_units      = 128,
)
print(config)

## Train


In [ ]:
from deepTechno.training.train_lstm import run_lstm_training
model, history = run_lstm_training(config)

## Loss curves


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, key in zip(axes, ['loss', 'pitch_loss', 'step_loss']):
    if key in history.history:
        ax.plot(history.history[key], label='train')
        val_key = f'val_{key}'
        if val_key in history.history:
            ax.plot(history.history[val_key], label='val')
        ax.set_title(key)
        ax.legend()
plt.tight_layout()
plt.show()

## Generate & play


In [ ]:
import pandas as pd, numpy as np
from deepTechno.generation.generate_lstm import generate_midi_lstm

df = pd.read_csv(ALL_NOTES)
seed = df[['pitch', 'step', 'duration']].values[:config.seq_length]

out_path = '/content/generated_lstm.mid'
generate_midi_lstm(
    model, seed, out_file=out_path,
    num_predictions=120, temperature=2.0,
    seq_length=config.seq_length, vocab_size=config.vocab_size,
)
print('Saved:', out_path)

In [ ]:
# In-browser playback via FluidSynth (requires Colab audio display)
from IPython.display import Audio
try:
    import subprocess
    subprocess.run(['apt-get', 'install', '-qq', 'fluidsynth'], check=True)
    subprocess.run([
        'fluidsynth', '-ni', '/usr/share/sounds/sf2/default-GM.sf2',
        out_path, '-F', '/content/lstm_out.wav', '-r', '44100'
    ], capture_output=True)
    display(Audio('/content/lstm_out.wav'))
except Exception as e:
    print('Audio playback unavailable:', e)
    print('Download the MIDI manually from Files panel.')